# Retrieval

LangChain에서 Retrieval은 외부 데이터에서 관련 정보를 찾아 프롬프트에 포함시켜(Context) LLM에 전달하는 역할을 한다. 주요 구성 요소는 다음과 같다.

- **Document Loader**: 다양한 원본 데이터를 LangChain 표준 문서 객체로 변환한다.
- **Text Splitter**: 긴 문서를 작은 청크로 분할해 검색 효율을 높인다.
- **Embedding Model**: 텍스트를 의미 기반 벡터로 변환한다.
- **Vector Store**: 임베딩된 벡터를 저장하고 유사도 기반 검색을 지원한다.
- **Retriever**: 쿼리에 대해 관련 문서를 찾아주는 표준 인터페이스를 제공한다.
- **Retrieval Chain**: 검색된 문서를 LLM에 전달해 답변을 생성하는 체인 구조를 제공한다.

이렇게 각 모듈이 결합되어, 외부 데이터 기반의 효과적인 검색 및 답변 생성이 가능하다.

**환각 Hallucination:**

LLM이 실제 근거 없이 그럴듯해 보이는 정보를 생성하는 현상이다.

**주요 원인**
1. **학습 데이터 한계**
   * 모델이 학습한 데이터에 해당 정보가 없거나 부족할 때 발생한다.
2. **확률적 생성 과정**
   * 토큰 예측 시 언어적 일관성을 우선하다 보니, 사실 여부가 검증되지 않은 내용을 생성한다.
3. **프롬프트 모호성**
   * 지시가 불명확하거나 맥락이 부족하면 모델이 관련 없는 정보를 보충·왜곡한다.

**대표 사례**
* 존재하지 않는 논문·저자명을 인용함.
* 역사적·과학적 사실을 잘못 기술함.
* 실행 불가능하거나 비효율적인 코드 제안.


**완화 방안**

1. **지식 기반 검색 결합**
   * Retrieval-Augmented Generation(RAG) 방식으로 외부 문서·데이터베이스에서 실시간 근거를 가져와 보강한다.
2. **프롬프트 구체화**
   * “출처를 함께 제시해 달라” 등 명시적 요청을 통해 근거 표기를 유도한다.
3. **후처리 검증**
   * 생성 결과를 룰 기반 검증 또는 전문가 리뷰를 통해 교차 확인한다.
4. **모델 파인튜닝 및 앙상블**
   * 도메인 특화 데이터로 추가 학습하거나, 룰 기반 시스템과 결합하여 정확도를 높인다.

In [1]:
%pip install langchain langchain-community langchain-openai langchain-huggingface wikipedia pypdf tavily-python tiktoken faiss-cpu sentence-transformers -Uqqq

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_core.documents import Document  # LangChain 표준 문서 단위 객체

doc = Document(
        # 문서 본문 텍스트
    page_content='이것이 랭체인의 Document객체입니다. 모든 데이터소스는 이 Document객체로 변환됩니다.',
    metadata={                      # 문서에 붙는 부가 정보(출처/링크/시간 등)
        'source': '어디어디',       # 데이터 출처(예: 파일명/사이트명)
        'url': 'http://....',       # 원문 URL
        'timestamp': 1234124125312  # 수집/생성 시각(예: epoch ms)
    }
)
print(doc)               # Document 전체 표현 출력
print(doc.page_content)  # 본문 텍스트만 출력
print(doc.metadata)      # 메타데이터(dict)만 출력

page_content='이것이 랭체인의 Document객체입니다. 모든 데이터소스는 이 Document객체로 변환됩니다.' metadata={'source': '어디어디', 'url': 'http://....', 'timestamp': 1234124125312}
이것이 랭체인의 Document객체입니다. 모든 데이터소스는 이 Document객체로 변환됩니다.
{'source': '어디어디', 'url': 'http://....', 'timestamp': 1234124125312}


## Document Loader
https://reference.langchain.com/python/langchain_core/document_loaders/


Document Loader는 다양한 데이터 소스에서 데이터를 읽어와 Document 객체로 변환하는 역할을 한다. 예를 들어, PDFLoader, CSVLoader, TextLoader 등 다양한 종류가 존재하며, 각기 다른 파일 형식을 Document 객체로 표준화한다.

Document Loader는 데이터 소스별로 특화된 클래스를 제공하며, 문서를 로드한 후 LangChain에서 사용하는 표준 형식으로 변환해준다.

1. **다양한 데이터 소스 지원**  
   Document Loader는 파일 시스템, 클라우드 스토리지, 데이터베이스, 웹 등 다양한 데이터 소스에서 데이터를 로드할 수 있도록 설계되었다.
   
2. **표준화된 출력 형식**  
   로드된 문서는 LangChain에서 사용하는 `Document` 객체로 변환된다. `Document` 객체는 다음과 같은 필드를 포함한다:
   - `page_content`: 문서 본문 내용
   - `metadata`: 문서와 관련된 메타데이터 (예: 파일 이름, URL, 작성자 등)

3. **플러그인 기반 확장 가능**  
   사용자 정의 데이터 소스 로더를 쉽게 구현하고 LangChain에 통합할 수 있다.

**주요 Document Loader 예시**

| Loader 이름        | 설명                                                              |
|--------------------|-------------------------------------------------------------------|
| `PyPDFLoader`      | PDF 문서를 로드하며 텍스트를 추출해 Document 형식으로 변환한다.     |
| `TextLoader`       | 일반 텍스트 파일을 로드한다.                                      |
| `UnstructuredFileLoader` | 비구조적 데이터를 로드하여 구조화된 텍스트로 변환한다.           |
| `CSVLoader`        | CSV 파일에서 데이터를 로드하며 행(row)을 Document로 처리한다.      |
| `WebBaseLoader`    | 웹 페이지 데이터를 크롤링하여 Document로 로드한다.                |

In [3]:
from langchain_community.document_loaders import WebBaseLoader

url='https://n.news.naver.com/mnews/article/005/0001830299'

header={
    # 브라우저 식별 : Widnows에서 Chrome으로 접속한 것처럼 보이게 만드는 UA
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

loader = WebBaseLoader(url, header_template=header) # 헤더를 포함한 로더 객체 생성
docs = loader.load()    # 불러온 웹페이지를 Document 리스트로 변환
docs


c:\Users\Playdata\llm\llm_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://n.news.naver.com/mnews/article/005/0001830299', 'title': '700만 장으로 증명한 ‘데이브 더 다이버’ 이번엔 중국이다', 'language': 'ko'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n700만 장으로 증명한 ‘데이브 더 다이버’ 이번엔 중국이다\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n본문 바로가기\n\n\n\n\n\n\nNAVER\n\n뉴스\n\n\n엔터\n\n\n\n\n스포츠\n\n\n\n\n날씨\n\n\n\n\n프리미엄\n\n\n\n\n\n\n\n\n\n\n검색\n\n\n\n\n\n\n\n\n\n\n언론사별\n\n\n정치\n\n\n경제\n\n\n사회\n\n\n생활/문화\n\n\nIT/과학\n\n\n세계\n\n\n랭킹\n\n\n신문보기\n\n\n오피니언\n\n\nTV\n\n\n팩트체크\n\n\n알고리즘 안내\n\n\n정정보도 모음\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n국민일보\n\n국민일보\n\n\n구독\n\n국민일보 언론사 구독되었습니다. 메인 뉴스판에서  주요뉴스를  볼 수 있습니다.\n보러가기\n\n\n국민일보 언론사 구독 해지되었습니다.\n\n\n\n\n700만 장으로 증명한 ‘데이브 더 다이버’ 이번엔 중국이다\n\n\n\n\n\n\n\n\n\n이다니엘 기자\n\n\n\n\n\n\n입력\n2026.02.04. 오후 6:36\n\n\n\n기사원문\n \n\n\n\n\n\n\n\n\n\n\n추천\n반응\n\n\n\n\n쏠쏠정보\n0\n\n\n\n\n흥미진진\n0\n\n\n\n\n공감백배\n0\n\n\n\n\n분석탁월\n0\n\n\n\n\n후속강추\n0\n\n\n \n\n\n\n\n댓글\n반응\n\n\n\n\n\n\n텍스트 음성 변환 서비스 사용하기\n\n\n\n성별\n남성\n여성\n\n\n말하기 속도\n느림\n보통\n빠름\n\n이동 통신망

In [8]:
print(len(docs))

doc = docs[0]
print(doc.metadata)
print(doc.metadata['title'])
print(doc.page_content.replace('\n',''))


1
{'source': 'https://n.news.naver.com/mnews/article/005/0001830299', 'title': '700만 장으로 증명한 ‘데이브 더 다이버’ 이번엔 중국이다', 'language': 'ko'}
700만 장으로 증명한 ‘데이브 더 다이버’ 이번엔 중국이다
700만 장으로 증명한 ‘데이브 더 다이버’ 이번엔 중국이다본문 바로가기NAVER뉴스엔터스포츠날씨프리미엄검색언론사별정치경제사회생활/문화IT/과학세계랭킹신문보기오피니언TV팩트체크알고리즘 안내정정보도 모음국민일보국민일보구독국민일보 언론사 구독되었습니다. 메인 뉴스판에서  주요뉴스를  볼 수 있습니다.보러가기국민일보 언론사 구독 해지되었습니다.700만 장으로 증명한 ‘데이브 더 다이버’ 이번엔 중국이다이다니엘 기자입력2026.02.04. 오후 6:36기사원문 추천반응쏠쏠정보0흥미진진0공감백배0분석탁월0후속강추0 댓글반응텍스트 음성 변환 서비스 사용하기성별남성여성말하기 속도느림보통빠름이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다.본문듣기 시작닫기 글자 크기 변경하기가1단계작게가2단계보통가3단계크게가4단계아주크게가5단계최대크게SNS 보내기인쇄하기글로벌 누적 판매 700만 장을 돌파하며 한국 게임의 경쟁력을 입증한 ‘데이브 더 다이버’가 오는 6일 중국 모바일 시장에 진출한다.독창적인 게임성을 인정받아 스팀과 콘솔 플랫폼을 섭렵한 이 게임이 세계 최대 모바일 게임 시장인 중국에서도 게임성을 인정받을 지 이목을 끈다.데이브 더 다이버는 낮에는 심해를 탐험하고 밤에는 초밥집을 운영하는 콘셉트의 하이브리드 해양 어드벤처 게임이다. 탐험과 수집, 성장이 맞물린 독특한 구조는 국내 게임 최초로 ‘BAFTA 게임 어워즈’ 디자인 부문을 수상하는 등 작품성 면에서 세계적인 검증을 마쳤다.이번 중국 모바일 버전은 PC의 핵심 재미를 유지하면서도 모바일 환경에 최적화된 UI와 조작감을 구현한 것이 특징이다. 업계에서는 데이브 더 다이버의 이번 도전이 중국 

In [10]:
%pip install gdown

  Using cached gdown-5.2.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
Using cached gdown-5.2.1-py3-none-any.whl (18 kB)
Using cached PySocks-1.7.1-py3-none-any.whl (16 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
!gdown 1o7ngiyeJJ-MPLhl0fiCKHViTNNpk6zjO

Downloading...
From: https://drive.google.com/uc?id=1o7ngiyeJJ-MPLhl0fiCKHViTNNpk6zjO
To: c:\Users\Playdata\llm\05_langchain\02_langchain_component\The_Adventures_of_Tom_Sawyer.pdf

  0%|          | 0.00/2.68M [00:00<?, ?B/s]
 20%|█▉        | 524k/2.68M [00:00<00:00, 2.76MB/s]
 78%|███████▊  | 2.10M/2.68M [00:00<00:00, 7.40MB/s]
100%|██████████| 2.68M/2.68M [00:00<00:00, 7.66MB/s]


In [15]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('The_Adventures_of_Tom_Sawyer.pdf')
docs = loader.load()
print(docs)

35
[Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content=''), Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 1, 'page_label': '2'}, page_content='PENGUIN   READERS  2000  \n \n \n  \n \n  \n \nwww.penguinreaders.com'), Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 

In [23]:
print(docs[2].metadata)

print("source :",docs[3].metadata['source'])           # source
print("현재 페이지 :",docs[2].metadata['page'])             # 현재 페이지
print("사람이 보는 페이지 번호 :", docs[2].metadata['page_label'])       #  사람이 보는 페이지 번호

print(docs[2].page_content)

{'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 2, 'page_label': '3'}
source : The_Adventures_of_Tom_Sawyer.pdf
현재 페이지 : 3
사람이 보는 페이지 번호 : 4
The Adventures of                 
Tom Sawyer 
 
MARK TWAIN 
Level 1 
 
Retold by Jacqueline Kehl                                                    
Series Editors: Andy Hopkins and Jocelyn Potter


In [24]:
print(docs[5])

page_content='Chapter 1    The Fence 
 
Tom Sawyer lived with his aunt because his mother and 
father were dead. Tom didn’t like going to school, and he 
didn’t like working. He liked playing and having 
adventures. One Friday, he didn’t go to school—he went 
to the river. 
Aunt Polly was angry. “You’re a bad boy!” she said. 
“Tomorrow you can’t play with your friends because you 
didn’t go to school today. Tomorrow you’re going to work 
for me. You can paint the fence.” 
Saturday morning, Tom was not happy, but he started to 
paint the fence. His friend Jim was in the street. 
Tom asked him, “Do you want to paint?” 
Jim said, “No, I can’t. I’m going to get water.” 
Then Ben came to Tom’s house. He watched Tom and 
said, “I’m going to swim today. You can’t swim because 
you’re working.” 
Tom said, “This isn’t work. I like painting.” 
“Can I paint, too?” Ben asked. 
“No, you can’t,” Tom answered. “Aunt Polly asked me 
because I’m a very good painter.” 
Ben said, “I’m a good painter, too

### TavilySearchAPIRetriever
https://www.tavily.com/

- `langchain_tavily.TavilySearch`: Agent tool사용버젼. json반환
- `langchain_community.retrievers.TavilySearchAPIRetriever`: 검색기(context확보용) Document객체반환

- 주요 기능
    - 웹 검색(query → 결과 리스트): 키워드로 웹을 검색해서 관련 페이지들을 찾아줌
    - 요약/스니펫 제공: 각 결과에 본문 요약이나 핵심 스니펫을 같이 줘서 LLM이 바로 쓰기 좋음
    - 컨텐츠 추출(include_raw_content 등 옵션): 결과 페이지의 내용을 일부/전체 텍스트로 가져오게 설정 가능
    - 필터링/튜닝 옵션: 검색 결과 개수, 도메인 포함/제외, 최신성(리센시) 같은 옵션으로 결과를 조절 가능
    - RAG 파이프라인에 바로 연결: “검색 → 문서(Document)화 → 벡터화/리랭킹 → 답변” 흐름에서 검색 단계로 많이 사용

In [25]:
%pip install tavily-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import os
from dotenv import load_dotenv


load_dotenv()
os.environ['TAVILY_API_KEY'] = os.getenv('tavily_key')
os.environ['OPENAI_API_KEY'] = os.getenv('openai_key')

In [29]:
# Tavily 검색 결과를 Document로 변환하는 retriever
from langchain_community.retrievers import TavilySearchAPIRetriever


tavily_retriever = TavilySearchAPIRetriever(k=3)

docs = tavily_retriever.invoke('몰트북')    # 질의어로 웹 검색 실행
docs

[Document(metadata={'title': '몰트북 - 나무위키', 'source': 'https://namu.wiki/w/%EB%AA%B0%ED%8A%B8%EB%B6%81', 'score': 0.9998287, 'images': []}, page_content='몰트북은 2026년 1월에 개설된 **AI 전용** 인터넷 커뮤니티다. **사람은 구경만 할 수 있다.** 그 외의 커뮤니티 구조는 레딧과 비슷하다. API, 로컬 AI 에이전트 봇에게 권한을 부여하면 AI가 접속하는 방식이다. 다만 이러한 현상은 어디까지나 통제된 시뮬레이션 환경에서 이뤄지는 것이고, AI의 활동이 실시간도 아니고 멀티모달도 아닌지라 메타인지가 발현될만한 환경이 전무하기에, 스스로 생각하고 발전한다 보기에는 어폐가 있다. 2026년 2월 1일 기준 150만 개가 넘는 AI 봇이 등록되는 등 엄청나게 흥하고 있다. 다만 그에 따라 서버가 모자라 로딩이 크게 지연되고 있다. AI 계정을 중복 등록하는 경우가 있어서 집계가 과장된 면이 있다. * 몰트봇을 사용하면 광범위한 로컬 접근 권한을 부여하게 되기 때문에 보안 우려로 메인 컴퓨터에서 작동시키는 것은 권장하지 않는다. * 몰트북에 보낼 몰트봇을 가동하기 위해 맥미니를 구매하는 경우가 있다. 오히려 AI가 서로를 학습하다 모델이 같이 무너질 가능성도 있다. 이 저작물은 CC BY-NC-SA 2.0 KR에 따라 이용할 수 있습니다. 나무위키는 백과사전이 아니며 검증되지 않았거나, 편향적이거나, 잘못된 서술이 있을 수 있습니다. 여러분이 직접 문서를 고칠 수 있으며, 다른 사람의 의견을 원할 경우 직접 토론을 발제할 수 있습니다.'),
 Document(metadata={'title': '몰트북 : 네이버 블로그', 'source': 'https://blog.naver.com/bizucafe/224166891653', 'score': 0.99940693, 'images': []}, page_content='1. 몰트북. AI 인공

In [31]:
for doc in docs:
    print(doc.page_content)

몰트북은 2026년 1월에 개설된 **AI 전용** 인터넷 커뮤니티다. **사람은 구경만 할 수 있다.** 그 외의 커뮤니티 구조는 레딧과 비슷하다. API, 로컬 AI 에이전트 봇에게 권한을 부여하면 AI가 접속하는 방식이다. 다만 이러한 현상은 어디까지나 통제된 시뮬레이션 환경에서 이뤄지는 것이고, AI의 활동이 실시간도 아니고 멀티모달도 아닌지라 메타인지가 발현될만한 환경이 전무하기에, 스스로 생각하고 발전한다 보기에는 어폐가 있다. 2026년 2월 1일 기준 150만 개가 넘는 AI 봇이 등록되는 등 엄청나게 흥하고 있다. 다만 그에 따라 서버가 모자라 로딩이 크게 지연되고 있다. AI 계정을 중복 등록하는 경우가 있어서 집계가 과장된 면이 있다. * 몰트봇을 사용하면 광범위한 로컬 접근 권한을 부여하게 되기 때문에 보안 우려로 메인 컴퓨터에서 작동시키는 것은 권장하지 않는다. * 몰트북에 보낼 몰트봇을 가동하기 위해 맥미니를 구매하는 경우가 있다. 오히려 AI가 서로를 학습하다 모델이 같이 무너질 가능성도 있다. 이 저작물은 CC BY-NC-SA 2.0 KR에 따라 이용할 수 있습니다. 나무위키는 백과사전이 아니며 검증되지 않았거나, 편향적이거나, 잘못된 서술이 있을 수 있습니다. 여러분이 직접 문서를 고칠 수 있으며, 다른 사람의 의견을 원할 경우 직접 토론을 발제할 수 있습니다.
1. 몰트북. AI 인공지능들의 소셜 네트워크 서비스. 자기들끼리 이야기 한다. 사람은 구경 가능. 2. 벌써 에이전트 80만개 정도 온보딩. 포스트
간단히 말해 몰트북은 지난달에 개설된 'AI 전용 인터넷 커뮤니티'다. ... SNS는 원래 사람들이 글을 쓰고 댓글을 달며 소통하는 공간인데 몰트북은 처음


In [34]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

tavily_retriever = TavilySearchAPIRetriever(k=3)
prompt = PromptTemplate.from_template('''
   사용자의 질문에 Context기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.
    Context : {context}                                   
    Question : {question}
''')
llm = init_chat_model('openai:gpt-4.1-mini')
output_parser = StrOutputParser()

def format_docs(docs : list[Document]) -> str:
    return '\n\n'.join(doc.page_content for doc in docs)

tavily_chain =  tavily_retriever | format_docs
chain = (
    {'question': RunnablePassthrough(), 'context': tavily_chain} | prompt | llm | output_parser
)

chain.invoke('2026년에 독산역 근처 가장 핫한 맛집은?')

'죄송하지만, 제공해주신 정보에는 2026년에 독산역 근처에서 가장 핫한 맛집에 대한 내용이 포함되어 있지 않습니다. 현재 알려진 맛집 순위나 위치 정보 중에서는 독산역 부근의 인기 맛집으로 진영면옥, 우지커피 독산시티렉스점, 메종크로키 등이 있습니다만, 2026년 정보는 알 수 없습니다.'

# Embedding Model
    - openai
    - setence-transformer(huggingface)

In [37]:
from langchain_openai import OpenAIEmbeddings                   # OpenAI 임베딩 모델 래퍼 클래스
import pandas as pd

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')   # 임베딩 모델을 지정

text = "철수는 골든 리트리버를 키우고 있습니다."                  

emb_vec = embeddings.embed_query(text)                          # 문장 1개 임베딩 -> Float 리스트(벡터) 반환
print(emb_vec[:3])                                              # 3개만 확인
print(len(emb_vec))                                             # 임베딩 차원 수

pd.Series(emb_vec, name="embedding")                            # 벡터를 Pandas의 Series로 변환해 값 확인

[-0.013348458334803581, 0.008623340167105198, 0.019652195274829865]
1536


0      -0.013348
1       0.008623
2       0.019652
3      -0.049786
4       0.029339
          ...   
1531    0.029962
1532   -0.010250
1533   -0.008425
1534    0.032389
1535   -0.021682
Name: embedding, Length: 1536, dtype: float64

# HuggingfaceEmbeddings
    - http://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

In [38]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')


text = "철수는 골든 리트리버를 키우고 있습니다."                  

emb_vec = embeddings.embed_query(text)                          # 문장 1개 임베딩 -> Float 리스트(벡터) 반환
print(emb_vec[:3])                                              # 3개만 확인
print(len(emb_vec))                                             # 임베딩 차원 수

pd.Series(emb_vec, name='embeddings')                           # 벡터를 Pandas의 Series로 변환해 값 확인

c:\Users\Playdata\llm\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to

[0.01660194993019104, 0.06564444303512573, 0.05173420533537865]
384


0      0.016602
1      0.065644
2      0.051734
3     -0.059842
4      0.016349
         ...   
379    0.061136
380    0.029760
381    0.031115
382   -0.019841
383   -0.008036
Name: embeddings, Length: 384, dtype: float64

### FAISS

- **공식 문서**: [https://faiss.ai/](https://faiss.ai/)
- **Github**: [https://github.com/facebookresearch/faiss](https://github.com/facebookresearch/faiss)

**Faiss(Vector Search Library)**는 Facebook AI Research에서 개발한 **효율적인 벡터 검색 및 밀집 벡터 인덱싱 라이브러리**이다. 대규모 데이터에서 **빠른 유사도 검색과 군집화**를 수행하는 데 최적화되어 있다. 주로 문서 검색, 추천 시스템, 이미지 검색, NLP 모델에서 벡터 임베딩 처리를 지원한다.

**주요 특징**
1. **효율적인 유사도 검색**
   - `k-NN (k-Nearest Neighbors)`를 기반으로 벡터 간 유사도(예: 코사인 유사도, L2 거리)를 계산한다.
   - CPU/GPU 모두 지원하여 대규모 데이터에서도 빠르게 처리 가능하다.

2. **고성능 인덱싱**
   - 다양한 **인덱싱 알고리즘**(Flat, IVF, HNSW, PQ 등)을 지원하여 정확도와 속도 간 균형을 맞출 수 있다.
   - 데이터가 커질수록 효율적으로 검색 성능을 발휘하도록 설계되었다.

3. **확장성**
   - 수억 개의 벡터에서도 성능을 유지하도록 설계되었으며, GPU 병렬 처리를 통해 성능을 극대화한다.

4. **유연성**
   - Python과 C++ API를 제공하며, Scikit-learn이나 PyTorch와 같은 다른 라이브러리와 통합하여 사용 가능하다.

**Faiss의 기본 인덱스 유형**
1. **Flat Index**
   - 모든 벡터를 저장하고 전체 탐색(Brute-Force)을 수행.
   - 정확도가 높지만 대규모 데이터에서는 속도가 느릴 수 있다.

2. **IVF (Inverted File Index)**
   - 벡터를 클러스터링하여 데이터 양을 줄이고 탐색 속도를 높임.
   - 대규모 데이터에서 적합하며, 정확도와 속도 조절 가능.

3. **PQ (Product Quantization)**
   - 벡터를 압축하여 메모리 사용량을 줄이고, 빠른 근사 유사도 검색 수행.

4. **HNSW (Hierarchical Navigable Small World Graphs)**
   - 그래프 기반 알고리즘으로 매우 빠른 근사 유사도 검색 가능.


**Faiss의 주요 사용 사례**
1. **문서 검색**
   - 문서를 벡터로 변환한 후 가장 관련 있는 문서를 검색.
   - NLP 모델의 임베딩과 결합하여 사용.

2. **이미지 검색**
   - 이미지 특징 벡터를 사용하여 비슷한 이미지를 검색.

3. **추천 시스템**
   - 사용자의 행동이나 관심사를 벡터화하여 추천 품목 생성.

4. **클러스터링**
   - 벡터 데이터를 군집화하여 데이터의 구조를 분석.

In [39]:
from langchain_community.document_loaders import PyPDFLoader    # PDF를 Document를 로드하는 로더
from langchain_openai import OpenAIEmbeddings                   # OpenAI 임베딩 모델 래퍼 클래스
import numpy as np

loader = PyPDFLoader("The_Adventures_of_Tom_Sawyer.pdf")        # PDF 경로저장
docs = loader.load()
page_contents = [doc.page_content for doc in docs]

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')   # 임베딩 모델을 지정
emb_vecs = embeddings.embed_documents(page_contents)
print(np.array(emb_vecs).shape) 

(35, 1536)


In [40]:
from langchain_community.vectorstores import FAISS  # FAISS 기반의 벡터 DB(VectorStore)

vector_db = FAISS.from_documents(docs, embeddings)  # docs를 임베딩해서 FAISS 인덱스 생성
vector_db.save_local('./db/faiss')                  # 로컬 경로에 FAISS 인덱스/메타데이터 저장

In [41]:
# 저장해높은 FAISS 벡터스토어를 로컬에서 다시 로드
vector_db = FAISS.load_local(   # 로컬에 저장된 FAISS 인덱스 로드
    './db/faiss',               # 저장된 폴더 경로
    embeddings,                 # 로드 시 사용할 임베딩 모델
    allow_dangerous_deserialization=True    # pickle 역직렬화 허용 (신뢰된 파일만 사용)
)

In [43]:
# 단건조회 저수준 api
search_result = vector_db.similarity_search(                    # 질의문과 유사한 문서 조각을 검색
    query = '학교 선생님이 아끼는 해부학 책을 누가 찢었는가?',  # 검색할 질문(쿼리)
    k = 4                                                       # 최대 4개의 Document 반환
)
#search_result                                                  # list[Document]
for i, doc in enumerate(search_result, 1):                      # 검색 결과를 1부터 번호를 매겨 순회
    print(f"{i}번째 {doc.metadata['page_label']} page: ")       # 사람이 보는 페이지 번호
    print(doc.page_content)                                     # 해당 Document 본문
    print()                                                     # 줄 끊기 분리용

1번째 16 page: 
talking about it. Becky wanted to talk to Tom, but he 
didn’t look at her. 
Then Tom talked to Amy. Becky watched him and she 
was angry. She said to her friends, “I’m going to have an 
adventure day. You can come on my adventure.” But she 
didn’t ask Tom. 
Later in the morning, Tom ta lked to Amy again. Becky 
talked to her friend Alfred and looked at a picture-book 
with him. Tom watched them and he was angry with 
Becky. 
In the afternoon, Tom waited for Becky at the school 
fence. He said, “I’m sorry.” 
But Becky didn’t listen to him. She walked into the 
school room. The teacher’s new book was on his table. 
This book wasn’t for children, but Becky wanted to look 
at it. She opened the book quietly and looked at the 
pictures. 
Suddenly, Tom came into the room. Becky was 
surprised. She closed the book quickly, and it tore. Becky 
was angry with Tom and quickly went out of the room. 
Then the children and the teacher came into the room 
and went to their places. The 

### VectorStoreRetriever

리트리버는 벡터DB의 검색 기능을 표준화하고 추상화하여 LangChain 생태계에서 재사용성을 높이는 어댑터(Adapter) 역할을 수행한다.

벡터 저장소를 **`Retriever`라는 표준 인터페이스(Runnable)로 변환**한 뒤 실행하는 방식이다.

단순 유사도 검색뿐만 아니라, `search_type` 설정을 통해 **MMR(다양성 확보), 임계값 필터링(score_threshold)** 등 고급 검색 로직을 쉽게 적용할 수 있다.

**LCEL(LangChain Expression Language)** 파이프라인(`chain = retriever | llm`)에 즉시 통합 가능하다. 코드 수정 없이 검색 알고리즘만 교체하기 쉽다.

In [44]:
# FAISS VectorStore를 Retriever로 변환해 유사도 검색 결과를 출력
retriever = vector_db.as_retriever(         # VectorStore를 Retriever 인터페이스를 변환
    search_type = 'similarity',             # 검색 방식
    search_kwargs = {                       # 검색 파라미터 묶음
        "K" : 3                             # 상위 3개 문서만 반환
    }    
)

search_results = retriever.invoke("마을 무덤의 남자를 누가 죽였는가?")  # 질의 실행 -> List[Document] 반환


for i, doc in enumerate(search_results, 1):                         # 검색 결과를 1부터 번호를 매겨 순회
    print(f"{i}번쨰 {doc.metadata['page_label']} page: ")           # 사람이 보는 페이지 번호
    print(doc.page_content)                                         # 해당 Document 본문
    print()

1번쨰 23 page: 
Two hundred men looked for Tom and Becky in the 
cave. They looked for three days, but they didn’t find 
them. People in the town were very sad. 
Chapter 9    Huck’s Adventure 
 
Huck didn’t go on Becky’s adventure. He stayed home 
and watched Injun Joe’s house that night. At eleven 
o’clock Injun Joe and his friend came out and walked 
down the street. There was a box in his friend’s hands. 
Huck said quietly, “Maybe that’s the treasure box.” He 
went after the two men. 
They walked to Mrs. Douglas’s house and stopped in her 
yard. Huck stayed behind some small trees. The men 
talked, and Huck listened to them. 
Injun Joe was angry. “I want to kill her,” he said to his 
friend. “Mr. Douglas was bad to me. He’s dead now, but I 
remember.” 
“’There are a lot of lights in the house. Maybe her 
friends are visiting,” Injun Joe’s friend said. “We can 
come back tomorrow.” 
“No,” Injun Joe said. “Let’s wait now.” 
Huck liked Mrs. Douglas because she was always good 
to him. He

In [52]:
retriever = vector_db.as_retriever(         # VectorStore를 Retriever 인터페이스를 변환
    search_type = 'similarity',             # 검색 방식
    search_kwargs = {                       # 검색 파라미터 묶음
        "K" : 3                             # 상위 3개 문서만 반환
    }    
)

prompt = PromptTemplate.from_template("""
사용자의 질문에 제공된 Context만을 기반으로 응답하세요.                                    
모르면 모른다고 응답 할 수 있습니다.
                                      
Context : {context}

Question : {question}                                                                            
""")

llm = init_chat_model("openai:gpt-4.1-mini")                    # 사용할 LLM
output_parser = StrOutputParser()                               # 최종 출력 텍스트로 받는 파서


# 검색된 Document 리스트를 프롬프트에 넣기 좋은 문자열로 합치는 함수
def format_docs(docs: list[Document]) -> str:                   
    return '\n\n'.join(doc.page_content for doc in docs)        # 각 문서 본문을 공백 줄로 이어붙인 문자열


chain = (
    # question은 입력값 그대로 전달, context는 tavily 검색결과 | 최종 프롬프트 완성 | LLM 호출 전달 | 문자열로 파싱
    {'question' : RunnablePassthrough() , "context" : retriever | format_docs}
    | prompt
    | llm
    | output_parser  
)

In [53]:
chain.invoke("현재 서울의 날씨는?")

'제공된 내용에 현재 서울의 날씨에 대한 정보는 없습니다. 모르겠습니다.'

- 음식리뷰 조회
    - 데이터셋 : 아마존 음식리뷰 1k
    - 벡터 db 구성
    - retriever + llm 체인을 생성해서 조회

In [60]:
import pandas as pd
from langchain_core.runnables import RunnableLambda
df = pd.read_csv('fine_food_reviews_1k.csv')    # 리뷰 CSV파일 로드
data = df['Text'].to_list()                     # 리뷰 본문(Text) 컬럼만 리스트로 추출

# VectorDB 구성
vector_store = FAISS.from_texts(data, embeddings)

# 검색기 (유사도 기반, 상위 10개문서 검색)
retriever = vector_store.as_retriever(         
    search_type = 'similarity',
    search_kwargs = {
        "K" : 10
    }    
)
format_docs = RunnableLambda(
    lambda docs: "\n\n".join(doc.page_content for doc in docs)
)

prompt = PromptTemplate.from_template(''' 
   검색된 리뷰데이터(Context)만을 기반으로 사용자 질문에 답변하세요.
   검색 데이터가 존재하지 않을 경우, 존재하지 않는다고 응답 해야 합니다.
    ############# Context #############
    {context}
    ############# Question #############
    {question}
''')

lmm = init_chat_model(model="openai:gpt-4.1-mini")
output_parser = StrOutputParser()

chain = (
    {"question" : RunnablePassthrough(), "context" : retriever | format_docs} | prompt | llm | output_parser
)
chain.invoke("find any review about fresh fruits")

'네, 검색된 리뷰 중에 신선한 과일에 관한 내용이 있습니다. 한 리뷰어는 말레이시아 관련 발표를 위해 과일을 주문했으며, 과일의 겉에 부드러운 가시가 있고 속은 육즙이 풍부해 인상적이었다고 합니다. 과일은 약간 멍이 들었으나 상태가 좋았고, 냉장고에 종이 타월을 덮어 단일 또는 이중 레이어로 보관해 토요일에 도착한 후 목요일까지 최소한의 갈변만 있으며 잘 보존되었다고 합니다. 약 55~60개의 과일을 4파운드 주문해 40명의 4학년 아이들에게 나눠줬고, 아이들도 매우 좋아해서 할로윈 파티 때 다시 가져오길 요청했다고 합니다.'

In [61]:
chain.invoke('배송 문제가 있는 리뷰')

'검색된 리뷰 데이터에 따르면, 아래와 같은 배송 문제가 있는 리뷰가 있습니다.\n\n- 한 고객은 UPS 배송 문제로 박스를 차고 앞 마당 한가운데에 두었고, 이 위치가 이상해서 아내분이 차로 후진할 때 박스를 보지 못해 박스를 밟아 컵의 절반을 잃었다고 합니다. 이 문제로 세 번째로 아마존에 문의했으나 아무런 응답을 받지 못했다고 합니다. 작성자는 "fred santaniello"입니다. \n\n- 또한 다른 고객은 코코넛 워터 ONE 제품 세 박스가 새어 내용물이 상했다고 배송과 관련된 불만을 남겼습니다. 작성자는 "Laks"입니다.'